In [2]:
import cv2
import numpy as np
from retinaface import RetinaFace
import matplotlib.pyplot as plt

In [3]:
# Função para detectar rostos usando Haar Cascade
def detect_faces_haar(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = detector.detectMultiScale(gray, scaleFactor=1.15, minNeighbors=5, minSize=(50, 50))
    return faces

# Função para detectar rostos usando YOLO
def detect_faces_yolo(img):
    net = cv2.dnn.readNetFromDarknet("yolov3.cfg", "yolov3.weights")
    layer_names = net.getLayerNames()
    
    # Modificar esta linha para lidar com diferentes formatos de saída
    output_layers_indices = net.getUnconnectedOutLayers()
    
    # Se a saída for uma lista de um único valor, use o valor diretamente
    if isinstance(output_layers_indices, np.ndarray):
        output_layers_indices = output_layers_indices.flatten()
    
    output_layers = [layer_names[i - 1] for i in output_layers_indices]
    
    height, width = img.shape[:2]
    
    # Pre-processamento da imagem
    blob = cv2.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
    net.setInput(blob)
    outputs = net.forward(output_layers)
    
    boxes = []
    
    # Processar saídas e desenhar bounding boxes
    for output in outputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]
            if confidence > 0.5:  # Ajustar o threshold
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)
                boxes.append([x, y, w, h])
    
    return boxes

# Função para detectar rostos usando RetinaFace
def detect_faces_retinaface(img_path):
    faces = RetinaFace.detect_faces(img_path)
    boxes = []
    for key in faces.keys():
        identity = faces[key]
        facial_area = identity["facial_area"]
        x, y, w, h = facial_area[0], facial_area[1], facial_area[2] - facial_area[0], facial_area[3] - facial_area[1]
        boxes.append([x, y, w, h])
    return boxes


In [4]:
# Função para calcular o IoU entre dois bounding boxes
def calculate_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    # Coordenadas das interseções
    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)
    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)

    # Áreas das caixas
    box1_area = w1 * h1
    box2_area = w2 * h2

    # União
    union_area = box1_area + box2_area - inter_area

    # IoU
    iou = inter_area / union_area if union_area > 0 else 0
    return iou

# Função para desenhar os bounding boxes em uma imagem
def draw_boxes(img, boxes, color=(0, 255, 0)):
    for (x, y, w, h) in boxes:
        cv2.rectangle(img, (x, y), (x + w, y + h), color, 2)

# Função principal para realizar a comparação
def compare_face_detection(img_path):
    img = cv2.imread(img_path)

    # Haar Cascade
    haar_faces = detect_faces_haar(img.copy())
    draw_boxes(img, haar_faces, color=(255, 0, 0))  # Azul

    # YOLO
    yolo_faces = detect_faces_yolo(img.copy())
    draw_boxes(img, yolo_faces, color=(0, 0, 255))  # Vermelho

    # RetinaFace
    retina_faces = detect_faces_retinaface(img_path)
    draw_boxes(img, retina_faces, color=(0, 255, 0))  # Verde

    # Exibir imagem com bounding boxes
    cv2.imshow("Face Detection Comparison", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Calcular IoU
    print("IoU Scores:")
    haar_iou = np.mean([calculate_iou(hf, rf) for hf, rf in zip(haar_faces, retina_faces)])
    yolo_iou = np.mean([calculate_iou(yf, rf) for yf, rf in zip(yolo_faces, retina_faces)])

    print(f"Haar Cascade IoU (vs RetinaFace): {haar_iou:.4f}")
    print(f"YOLO IoU (vs RetinaFace): {yolo_iou:.4f}")


In [6]:
#testes
compare_face_detection('./people.jpg')

IoU Scores:
Haar Cascade IoU (vs RetinaFace): 0.0000
YOLO IoU (vs RetinaFace): 0.0001
